In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import average_precision_score

In [2]:
groundtruth = pd.read_csv("Noise_GroundTruth_and_TrainingLoss.csv")

foif = pd.read_csv("NoisyLabel_FOIF_Scores.csv")

tracin = pd.read_csv("NoisyLabel_TracIn_Scores.csv")

In [3]:
foif = foif.rename(columns={"Score":"FOIF_Score"})
tracin = tracin.rename(columns={"Score":"TracIn_Score"})

merged = (
    groundtruth
    .merge(
        foif[["Train_ID","FOIF_Score"]],
        on="Train_ID"
    )
    .merge(
        tracin[["Train_ID","TracIn_Score"]],
        on="Train_ID"
    )
)

print(merged.head())

   Train_ID  Clean_Label  Noisy_Label  is_noisy  Training_Loss  FOIF_Score  \
0         1            0            0         0       0.082718    0.162673   
1         2            1            1         0       0.141169    0.193420   
2         3            1            1         0       0.304665    0.245969   
3         4            1            1         0       0.599408    0.231110   
4         5            0            0         0       0.315456    0.204313   

   TracIn_Score  
0      0.006024  
1      0.015964  
2      0.001933  
3      0.018973  
4      0.012374  


In [4]:
random_baseline_seed = 42

rng = np.random.default_rng(random_baseline_seed)

merged["Random_Score"] = rng.random(len(merged))
# merged[["Train_ID", "Random_Score"]].to_csv(
#     "Random_Ranking.csv",
#     index=False
# )
print(merged)

      Train_ID  Clean_Label  Noisy_Label  is_noisy  Training_Loss  FOIF_Score  \
0            1            0            0         0       0.082718    0.162673   
1            2            1            1         0       0.141169    0.193420   
2            3            1            1         0       0.304665    0.245969   
3            4            1            1         0       0.599408    0.231110   
4            5            0            0         0       0.315456    0.204313   
...        ...          ...          ...       ...            ...         ...   
7995      7996            0            0         0       0.160876    0.323826   
7996      7997            0            1         1       0.454101    0.227448   
7997      7998            0            0         0       0.304615    0.318565   
7998      7999            0            0         0       0.159375    0.256325   
7999      8000            0            0         0       0.082865    0.130082   

      TracIn_Score  Random_

In [5]:
print(
    merged.groupby("is_noisy")[
        "FOIF_Score"
    ].mean()
)

print(
    merged.groupby("is_noisy")[
        "TracIn_Score"
    ].mean()
)

is_noisy
0    0.127046
1   -0.505098
Name: FOIF_Score, dtype: float64
is_noisy
0    0.006380
1   -0.022557
Name: TracIn_Score, dtype: float64


In [6]:
def precision_recall_at_k(
    y_true,
    scores,
    k_percentage,
    retrieve="top"
):
    """
    Parameters
    ----------
    y_true : binary ground truth
    scores : ranking scores
    k_percentage : e.g. 0.1 for Top/Bottom 10%
    retrieve : "top" or "bottom"
    """

    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = int(len(scores) * k_percentage)

    if retrieve == "top":
        selected = np.argsort(scores)[::-1][:k]

    elif retrieve == "bottom":
        selected = np.argsort(scores)[:k]

    else:
        raise ValueError("retrieve must be 'top' or 'bottom'")

    tp = y_true[selected].sum()

    precision = tp / k
    recall = tp / y_true.sum()

    return precision, recall

In [7]:
results = []

methods = {

    "Random":
    {
        "score": merged["Random_Score"],
        "retrieve": "bottom"
    },

    "Training Loss":
    {
        "score": merged["Training_Loss"],
        "retrieve": "top"
    },

    "FOIF":
    {
        "score": merged["FOIF_Score"],
        "retrieve": "bottom"
    },

    "TracIn":
    {
        "score": merged["TracIn_Score"],
        "retrieve": "bottom"
    }
}

In [8]:
for method, info in methods.items():

    scores = info["score"]

    retrieve = info["retrieve"]

    # AUPRC
    #
    # sklearn assumes larger score = positive
    #
    # Therefore we only negate FOIF/TracIn
    if retrieve == "bottom":
        auprc = average_precision_score(
            merged["is_noisy"],
            -scores
        )
    else:
        auprc = average_precision_score(
            merged["is_noisy"],
            scores
        )

    row = {

        "Method": method,

        "AUPRC": auprc
    }

    for p in [0.10,0.20,0.30]:

        precision, recall = precision_recall_at_k(

            merged["is_noisy"],

            scores,

            p,

            retrieve
        )

        row[f"Precision@{int(p*100)}"] = precision

        row[f"Recall@{int(p*100)}"] = recall

    results.append(row)

In [9]:
result_df = pd.DataFrame(results)

print(result_df.round(4))

          Method   AUPRC  Precision@10  Recall@10  Precision@20  Recall@20  \
0         Random  0.1919        0.1600     0.0800        0.1850     0.1850   
1  Training Loss  0.7376        0.8275     0.4138        0.6769     0.6769   
2           FOIF  0.7082        0.8062     0.4031        0.6688     0.6688   
3         TracIn  0.5971        0.7150     0.3575        0.5744     0.5744   

   Precision@30  Recall@30  
0        0.1925     0.2888  
1        0.5438     0.8156  
2        0.5350     0.8025  
3        0.4579     0.6869  
